## Check the GPU and connect Google Drive

In [8]:
!nvidia-smi

Mon Oct 12 05:50:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             31W /   70W |    2637MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
from google.colab import drive
import os

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/emotion-app'
os.makedirs(OUT, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Get the dataset from Kaggle

In [10]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_2a5588f4ac93d8a5b29bfb25a6377988'

In [11]:
!kaggle datasets download -d msambare/fer2013 -p /content/data --unzip
!ls /content/data

Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
100% 60.3M/60.3M [00:00<00:00, 124MB/s]

test  train


## Load and prepare the data

In [12]:
import json
import numpy as np
import torch, torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SIZE = 96
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Training images get random changes so the model handles real webcam variety
train_tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.RandomResizedCrop(SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Validation/test images are only resized, never randomized
eval_tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.Resize((SIZE, SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

root = '/content/data'
train_full = datasets.ImageFolder(f'{root}/train', train_tf)
val_full   = datasets.ImageFolder(f'{root}/train', eval_tf)
test_ds    = datasets.ImageFolder(f'{root}/test',  eval_tf)
classes = train_full.classes
print(classes)

# Hold out 10% of train for validation (picking the best epoch)
rng = np.random.RandomState(42)
idx = rng.permutation(len(train_full))
n_val = int(0.1 * len(idx))
val_idx, train_idx = idx[:n_val], idx[n_val:]

train_dl = DataLoader(Subset(train_full, train_idx), batch_size=128, shuffle=True, num_workers=2)
val_dl   = DataLoader(Subset(val_full, val_idx), batch_size=256, num_workers=2)
test_dl  = DataLoader(test_ds, batch_size=256, num_workers=2)

# Rare classes (like "disgust") get a higher weight in the loss
targets = np.array(train_full.targets)
counts = np.bincount(targets[train_idx], minlength=len(classes))
class_weights = torch.tensor(counts.sum() / (len(classes) * counts), dtype=torch.float).to(device)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


## Build the model

In [13]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.last_channel, len(classes))
model = model.to(device)

## Train

In [14]:
EPOCHS = 20
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, epochs=EPOCHS, steps_per_epoch=len(train_dl)
)

def accuracy(dl):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

best = 0
for epoch in range(EPOCHS):
    model.train()
    loss_sum = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        loss_sum += loss.item()
    val_acc = accuracy(val_dl)
    print(f'epoch {epoch+1}/{EPOCHS}  loss {loss_sum/len(train_dl):.3f}  val_acc {val_acc:.3f}')
    if val_acc > best:
        best = val_acc
        torch.save(model.state_dict(), f'{OUT}/best.pt')

epoch 1/20  loss 2.007  val_acc 0.380
epoch 2/20  loss 1.778  val_acc 0.464
epoch 3/20  loss 1.666  val_acc 0.554
epoch 4/20  loss 1.602  val_acc 0.556
epoch 5/20  loss 1.562  val_acc 0.592
epoch 6/20  loss 1.532  val_acc 0.599
epoch 7/20  loss 1.500  val_acc 0.576
epoch 8/20  loss 1.473  val_acc 0.620
epoch 9/20  loss 1.443  val_acc 0.621
epoch 10/20  loss 1.423  val_acc 0.620
epoch 11/20  loss 1.399  val_acc 0.629
epoch 12/20  loss 1.364  val_acc 0.652
epoch 13/20  loss 1.330  val_acc 0.661
epoch 14/20  loss 1.300  val_acc 0.657
epoch 15/20  loss 1.264  val_acc 0.672
epoch 16/20  loss 1.222  val_acc 0.675
epoch 17/20  loss 1.182  val_acc 0.684
epoch 18/20  loss 1.150  val_acc 0.690
epoch 19/20  loss 1.132  val_acc 0.692
epoch 20/20  loss 1.125  val_acc 0.690


## Test it honestly

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load(f'{OUT}/best.pt'))
model.eval()
preds, labels = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        labels += y.tolist()

print(classification_report(labels, preds, target_names=classes))
print(confusion_matrix(labels, preds))

              precision    recall  f1-score   support

       angry       0.58      0.63      0.61       958
     disgust       0.55      0.68      0.60       111
        fear       0.56      0.48      0.52      1024
       happy       0.88      0.87      0.87      1774
     neutral       0.63      0.68      0.65      1233
         sad       0.58      0.52      0.55      1247
    surprise       0.75      0.84      0.79       831

    accuracy                           0.68      7178
   macro avg       0.65      0.67      0.66      7178
weighted avg       0.68      0.68      0.68      7178

[[ 602   26   88   35   82  103   22]
 [  21   75    5    2    3    4    1]
 [ 137   10  492   24   97  155  109]
 [  21    5   28 1538   84   43   55]
 [  85    5   50   80  837  153   23]
 [ 145   11  163   50  214  643   21]
 [  20    5   58   25   18   10  695]]


## Save the settings Phase 3 needs

In [16]:
json.dump(
    {'classes': classes, 'size': SIZE, 'mean': MEAN, 'std': STD},
    open(f'{OUT}/meta.json', 'w')
)